## Residual mean reversion logic

The strategy is based on a simple mean reversion idea applied to the regression residual.

### Sell signal

If the current close is far above the regression estimate, the residual becomes strongly positive.

In this case, the model interprets the price as temporarily stretched to the upside and generates a short signal.

```text
High positive residual → price above estimated value → sell
```

### Buy signal

If the current close is far below the regression estimate, the residual becomes strongly negative.

In this case, the model interprets the price as temporarily stretched to the downside and generates a long signal.

```text
High negative residual → price below estimated value → buy
```

### Core interpretation


The strategy tries to capture the potential correction of extreme deviations between the actual close and the regression-implied close.


In [5]:
# ============================================================
# Outspoken Market Global
# Core idea:
# 1. Download OHLCV data from TradingView.
# 2. Split the data by row count:
#    - First N records = training set
#    - Remaining records = testing set
# 3. Fit a simple linear regression on the training set:
#    Close_t = beta0 + beta1 * Close_lag_t + error_t
# 4. Calculate the residual:
#    Residual_t = Close_t - Regression_t
# 5. Use extreme residuals as mean-reversion signals.
# 6. Test the signal on the next-period return.
# ============================================================

import pandas as pd
import numpy as np

from tvDatafeed import TvDatafeed, Interval

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import statsmodels.api as sm


def run_regression_residual_strategy(
    symbol="US500",
    exchange="CAPITALCOM",
    interval=Interval.in_daily,
    n_bars=5000,
    train_size=2000,
    threshold=40,
    trading_cost=0.01,
    show_model_summary=True
):
    """
    Run a regression residual strategy using TradingView data.

    Parameters
    ----------
    symbol : str
        TradingView symbol.
        Example: "US500", "BTCUSD", "EURUSD".

    exchange : str
        TradingView exchange.
        Example: "CAPITALCOM", "BINANCE", "FX".

    interval : tvDatafeed.Interval
        TradingView timeframe.
        Example: Interval.in_daily, Interval.in_4_hour, Interval.in_1_hour.

    n_bars : int
        Number of candles requested from TradingView.

    train_size : int
        Number of first records used for training.
        The remaining records are used for testing.

    threshold : float
        Fixed residual threshold used by the simple rule.

    trading_cost : float
        Trading cost in percentage points.
        Example:
        0.01 means 0.01%.

    show_model_summary : bool
        If True, prints the full statsmodels OLS summary.

    Returns
    -------
    dict
        Dictionary containing:
        - full cleaned dataset
        - training dataset
        - testing dataset
        - fitted regression model
        - residual summary
        - simple rule table
        - quartile rule table
        - final performance summary
    """

    # ========================================================
    # 1. Download data from TradingView
    # ========================================================

    # Initialize TradingView connection.
    #
    # If you have TradingView credentials, you can replace this with:
    #
    # tv = TvDatafeed(
    #     username="YOUR_USERNAME",
    #     password="YOUR_PASSWORD"
    # )
    #
    # Without credentials, tvDatafeed may still work, but access can be limited.
    tv = TvDatafeed()

    # Download historical OHLCV data from TradingView.
    df = tv.get_hist(
        symbol=symbol,
        exchange=exchange,
        interval=interval,
        n_bars=n_bars
    )

    # Validate that TradingView returned data.
    if df is None or df.empty:
        raise ValueError(
            "No data was returned by TradingView. "
            "Check the symbol, exchange, interval, or connection."
        )

    # The TradingView dataframe comes with datetime as the index.
    # We reset the index so datetime becomes a regular column.
    df = df.reset_index()

    # Standardize column names.
    #
    # tvDatafeed usually returns:
    # datetime, open, high, low, close, volume
    #
    # We rename them to a cleaner format.
    df = df.rename(columns={
        "datetime": "Date",
        "open": "Open",
        "high": "High",
        "low": "Low",
        "close": "Close",
        "volume": "Volume"
    })

    # Keep only the columns required for the analysis.
    #
    # We do not create an adjusted close column because TradingView data
    # does not follow the Yahoo Finance adjusted-price structure.
    # The strategy uses the TradingView Close directly.
    df = df[["Date", "Open", "High", "Low", "Close", "Volume"]].copy()

    # Convert date column to datetime format.
    df["Date"] = pd.to_datetime(df["Date"])

    # Remove missing values, sort by date, and reset index.
    df = df.dropna().sort_values("Date").reset_index(drop=True)

    # Check whether we have enough data for the requested training size.
    #
    # We need at least train_size records for training plus additional
    # observations for testing, after creating lag and lead variables.
    if len(df) <= train_size + 2:
        raise ValueError(
            f"Not enough data. You requested train_size={train_size}, "
            f"but only {len(df)} records were returned."
        )

    # ========================================================
    # 2. Calculate lag, lead, returns and target
    # ========================================================

    # Previous close.
    #
    # This is the explanatory variable used in the regression.
    # In mathematical terms:
    #
    # Close_lag_t = Close_{t-1}
    df["Close_lag"] = df["Close"].shift(1)

    # Next close.
    #
    # This is used to calculate the next-period target return.
    # In mathematical terms:
    #
    # Close_lead_t = Close_{t+1}
    df["Close_lead"] = df["Close"].shift(-1)

    # Remove missing values created by lag and lead.
    #
    # The first row has no previous close.
    # The last row has no next close.
    df = df.dropna().reset_index(drop=True)

    # Current return in percentage.
    #
    # This measures the return from the previous candle close to
    # the current candle close.
    #
    # Return_t = (Close_t / Close_{t-1} - 1) * 100
    df["Return"] = (df["Close"] / df["Close_lag"] - 1) * 100

    # Binary current return.
    #
    # 1 means current return was positive.
    # 0 means current return was zero or negative.
    df["Return_Bin"] = np.where(df["Return"] > 0, 1, 0)

    # Target return.
    #
    # This is the return the strategy will be tested against.
    #
    # The signal is generated at time t using information available at t.
    # The performance is measured from Close_t to Close_{t+1}.
    #
    # Target1_t = (Close_{t+1} / Close_t - 1) * 100
    df["Target1"] = (df["Close_lead"] / df["Close"] - 1) * 100

    # Binary target.
    #
    # 1 means the next candle return was positive.
    # 0 means the next candle return was zero or negative.
    df["Target1_Bin"] = np.where(df["Target1"] > 0, 1, 0)

    # ========================================================
    # 3. Split data into training and testing sets
    # ========================================================

    # Since we are using TradingView data, we split by row count.
    #
    # The first train_size observations are used for training.
    # The remaining observations are used for testing.
    #
    # Example:
    # train_size = 2000
    #
    # Rows 0 to 1999:
    # training set
    #
    # Rows 2000 onward:
    # testing set
    df_train = df.iloc[:train_size].copy()
    df_test = df.iloc[train_size:].copy()

    # Store date ranges for reporting.
    train_start = df_train["Date"].iloc[0]
    train_end = df_train["Date"].iloc[-1]

    test_start = df_test["Date"].iloc[0]
    test_end = df_test["Date"].iloc[-1]

    # Print a clear summary of the split.
    print("\nDataset information")
    print("===================")
    print(f"Asset: {symbol}")
    print(f"Exchange: {exchange}")
    print(f"Total records after cleaning: {len(df)}")
    print(f"Training records: {len(df_train)} | {train_start.date()} to {train_end.date()}")
    print(f"Testing records: {len(df_test)} | {test_start.date()} to {test_end.date()}")

    # ========================================================
    # 4. Fit the linear regression model
    # ========================================================

    # Regression model:
    #
    # Close_t = beta0 + beta1 * Close_lag_t + error_t
    #
    # This is not a machine learning model in the complex sense.
    # It is a simple statistical relationship between today's close
    # and yesterday's close.
    #
    # The purpose is not to directly forecast the next candle.
    # The purpose is to create a regression baseline and then study
    # the residual around that baseline.

    X_train = df_train[["Close_lag"]]
    y_train = df_train["Close"]

    # Add a constant term so the model can estimate the intercept beta0.
    X_train_const = sm.add_constant(X_train)

    # Fit OLS regression.
    model = sm.OLS(y_train, X_train_const).fit()

    # Print full regression summary if requested.
    if show_model_summary:
        print("\nOLS regression summary")
        print("======================")
        print(model.summary())

    # Extract regression coefficients.
    intercept = model.params["const"]
    beta_lag = model.params["Close_lag"]

    print("\nRegression coefficients")
    print("=======================")
    print(f"Intercept: {intercept:.6f}")
    print(f"Beta Close_lag: {beta_lag:.6f}")

    # ========================================================
    # 5. Calculate regression estimates and residuals
    # ========================================================

    # Regression estimate in the training set.
    #
    # Regression_t = beta0 + beta1 * Close_lag_t
    df_train["Regression"] = intercept + beta_lag * df_train["Close_lag"]

    # Training residual.
    #
    # Residual_t = Close_t - Regression_t
    #
    # Positive residual:
    # The actual close is above the regression estimate.
    #
    # Negative residual:
    # The actual close is below the regression estimate.
    df_train["Residual"] = df_train["Close"] - df_train["Regression"]

    # Apply the same regression equation to the testing set.
    #
    # Important:
    # The model is not refitted on the testing set.
    # We use the coefficients learned only from the training set.
    df_test["Regression"] = intercept + beta_lag * df_test["Close_lag"]

    # Testing residual.
    df_test["Residual"] = df_test["Close"] - df_test["Regression"]

    # ========================================================
    # 6. Calculate residual statistics from training data
    # ========================================================

    # The training residual distribution is used to define thresholds.
    #
    # This is important:
    # We do not use testing data to create the thresholds.
    # Otherwise, we would leak future information into the strategy.
    residual_summary = df_train["Residual"].describe()

    # First quartile.
    #
    # 25% of the training residuals are below this value.
    q1 = df_train["Residual"].quantile(0.25)

    # Third quartile.
    #
    # 75% of the training residuals are below this value.
    q3 = df_train["Residual"].quantile(0.75)

    # Quartile rule thresholds.
    #
    # These reproduce the logic from the original R code:
    #
    # Upper threshold = Q3 * 2
    # Lower threshold = Q1 * 0.5
    #
    # Interpretation:
    # If residual is above Q3 * 2, the price is considered very high
    # relative to the regression estimate.
    #
    # If residual is below Q1 * 0.5, the price is considered low
    # relative to the regression estimate.
    #
    # Important:
    # This rule is asymmetric.
    #
    # If Q1 is negative, multiplying Q1 by 0.5 makes the lower threshold
    # less negative, meaning long signals may be triggered more easily.
    upper_quartile_threshold = q3 * 2
    lower_quartile_threshold = q1 * 0.5

    print("\nTraining residual summary")
    print("=========================")
    print(residual_summary)

    print("\nResidual thresholds")
    print("===================")
    print(f"Fixed upper threshold: {threshold:.4f}")
    print(f"Fixed lower threshold: {-threshold:.4f}")
    print(f"Quartile upper threshold, Q3 * 2: {upper_quartile_threshold:.4f}")
    print(f"Quartile lower threshold, Q1 * 0.5: {lower_quartile_threshold:.4f}")

    # ========================================================
    # 7. Plot training residuals
    # ========================================================

    # This chart shows the residual behavior in the training set.
    #
    # It helps the user visually inspect whether the selected fixed
    # threshold makes sense for the asset and timeframe.
    fig_residual = go.Figure()

    fig_residual.add_trace(
        go.Scatter(
            x=df_train["Date"],
            y=df_train["Residual"],
            mode="lines",
            name="Training residual",
            line=dict(width=2)
        )
    )

    # Fixed upper threshold.
    fig_residual.add_hline(
        y=threshold,
        line_width=2,
        line_dash="dash",
        annotation_text=f"Upper threshold: {threshold}",
        annotation_position="top left"
    )

    # Fixed lower threshold.
    fig_residual.add_hline(
        y=-threshold,
        line_width=2,
        line_dash="dash",
        annotation_text=f"Lower threshold: {-threshold}",
        annotation_position="bottom left"
    )

    fig_residual.update_layout(
        title=f"Regression residuals in the training set<br>{symbol} | {exchange} | Outspoken Market",
        xaxis_title="Date",
        yaxis_title="Residual",
        template="plotly_white",
        width=1000,
        height=600
    )

    fig_residual.show()

    # ========================================================
    # 8. Simple residual rule
    # ========================================================

    # The simple rule uses a fixed threshold.
    #
    # Logic:
    #
    # If Residual > threshold:
    # The current close is far above the regression estimate.
    # The strategy assumes possible mean reversion and goes short.
    # Signal = -1
    #
    # If Residual < -threshold:
    # The current close is far below the regression estimate.
    # The strategy assumes possible mean reversion and goes long.
    # Signal = +1
    #
    # Otherwise:
    # The strategy stays out of the market.
    # Signal = 0

    df_test["Simple_Rule"] = np.where(
        df_test["Residual"] > threshold,
        -1,
        np.where(df_test["Residual"] < -threshold, 1, 0)
    )

    # Apply trading cost only when there is an active position.
    #
    # This is operationally cleaner than subtracting costs every period,
    # including periods when the strategy is out of the market.
    df_test["Simple_Cost"] = np.where(
        df_test["Simple_Rule"] != 0,
        trading_cost,
        0
    )

    # Strategy return.
    #
    # If signal = +1:
    # Strategy return = next-period market return minus cost.
    #
    # If signal = -1:
    # Strategy return = negative of next-period market return minus cost.
    #
    # If signal = 0:
    # Strategy return = 0.
    #
    # Formula:
    # Strategy_Return_t = Signal_t * Target1_t - Cost_t
    df_test["Simple_Strategy_Return"] = (
        df_test["Simple_Rule"] * df_test["Target1"] - df_test["Simple_Cost"]
    )

    # Cumulative strategy return.
    df_test["Simple_Strategy_Cumulative_Return"] = (
        df_test["Simple_Strategy_Return"].cumsum()
    )

    # ========================================================
    # 9. Simple rule directional table
    # ========================================================

    # This table compares the future market direction with the model signal.
    #
    # Rows:
    # Target1_Bin
    # 0 = next candle was down or flat
    # 1 = next candle was up
    #
    # Columns:
    # Simple_Rule
    # -1 = short
    #  0 = no position
    #  1 = long
    #
    # normalize="index" means each row sums to 100%.
    simple_table = pd.crosstab(
        df_test["Target1_Bin"],
        df_test["Simple_Rule"],
        normalize="index"
    ) * 100

    print("\nPercentage table: Target1_Bin vs Simple_Rule")
    print("============================================")
    print(simple_table.round(2))

    # ========================================================
    # 10. Plot simple rule cumulative return
    # ========================================================

    fig_simple = go.Figure()

    fig_simple.add_trace(
        go.Scatter(
            x=df_test["Date"],
            y=df_test["Simple_Strategy_Cumulative_Return"],
            mode="lines",
            name="Simple rule",
            line=dict(width=2)
        )
    )

    fig_simple.update_layout(
        title=f"Simple rule cumulative return<br>{symbol} | {exchange} | Costs included | Outspoken Market",
        xaxis_title="Date",
        yaxis_title="Cumulative return in %",
        template="plotly_white",
        width=1000,
        height=600
    )

    fig_simple.show()

    # ========================================================
    # 11. Quartile residual rule
    # ========================================================

    # The quartile rule is an adaptive version of the residual threshold.
    #
    # Instead of using a fixed number such as 490, it uses the residual
    # distribution from the training set.
    #
    # Original R logic:
    #
    # Sell when:
    # Residual > Q3 * 2
    #
    # Buy when:
    # Residual < Q1 * 0.5
    #
    # Stay out otherwise.
    #
    # This means:
    # - The upper threshold is based on the upper quartile of residuals.
    # - The lower threshold is based on the lower quartile of residuals.
    #
    # Important:
    # We preserve the original R logic here.
    # However, this rule is not perfectly symmetric.
    # For many assets, a more symmetric version could use Q1 * 2
    # or an IQR-based threshold.

    df_test["Quartile_Rule"] = np.where(
        df_test["Residual"] > upper_quartile_threshold,
        -1,
        np.where(df_test["Residual"] < lower_quartile_threshold, 1, 0)
    )

    # Apply trading cost only when there is an active position.
    df_test["Quartile_Cost"] = np.where(
        df_test["Quartile_Rule"] != 0,
        trading_cost,
        0
    )

    # Quartile strategy return.
    df_test["Quartile_Strategy_Return"] = (
        df_test["Quartile_Rule"] * df_test["Target1"] - df_test["Quartile_Cost"]
    )

    # Quartile strategy cumulative return.
    df_test["Quartile_Strategy_Cumulative_Return"] = (
        df_test["Quartile_Strategy_Return"].cumsum()
    )

    # ========================================================
    # 12. Quartile rule directional table
    # ========================================================

    quartile_table = pd.crosstab(
        df_test["Target1_Bin"],
        df_test["Quartile_Rule"],
        normalize="index"
    ) * 100

    print("\nPercentage table: Target1_Bin vs Quartile_Rule")
    print("==============================================")
    print(quartile_table.round(2))

    # ========================================================
    # 13. Plot comparison between simple and quartile rules
    # ========================================================

    fig_comparison = go.Figure()

    fig_comparison.add_trace(
        go.Scatter(
            x=df_test["Date"],
            y=df_test["Simple_Strategy_Cumulative_Return"],
            mode="lines",
            name="Simple rule",
            line=dict(width=2)
        )
    )

    fig_comparison.add_trace(
        go.Scatter(
            x=df_test["Date"],
            y=df_test["Quartile_Strategy_Cumulative_Return"],
            mode="lines",
            name="Quartile rule",
            line=dict(width=2)
        )
    )

    fig_comparison.update_layout(
        title=f"Strategy cumulative returns<br>{symbol} | {exchange} | Costs included | Outspoken Market",
        xaxis_title="Date",
        yaxis_title="Cumulative return in %",
        template="plotly_white",
        width=1000,
        height=600,
        legend_title="Strategy"
    )

    fig_comparison.show()

    # ========================================================
    # 14. Diagnostic chart
    # ========================================================

    # This chart helps explain the model visually.
    #
    # Panel 1:
    # Shows the actual close and the regression estimate.
    #
    # Panel 2:
    # Shows the residual, which is the actual input used by the rules.
    fig_diagnostic = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=(
            "Real close vs regression estimate",
            "Regression residual in the testing set"
        )
    )

    # Real close.
    fig_diagnostic.add_trace(
        go.Scatter(
            x=df_test["Date"],
            y=df_test["Close"],
            mode="lines",
            name="Close",
            line=dict(width=2)
        ),
        row=1,
        col=1
    )

    # Regression estimate.
    fig_diagnostic.add_trace(
        go.Scatter(
            x=df_test["Date"],
            y=df_test["Regression"],
            mode="lines",
            name="Regression estimate",
            line=dict(width=2)
        ),
        row=1,
        col=1
    )

    # Residual.
    fig_diagnostic.add_trace(
        go.Scatter(
            x=df_test["Date"],
            y=df_test["Residual"],
            mode="lines",
            name="Residual",
            line=dict(width=2)
        ),
        row=2,
        col=1
    )

    # Fixed threshold lines on residual panel.
    fig_diagnostic.add_hline(
        y=threshold,
        line_width=1,
        line_dash="dash",
        row=2,
        col=1
    )

    fig_diagnostic.add_hline(
        y=-threshold,
        line_width=1,
        line_dash="dash",
        row=2,
        col=1
    )

    fig_diagnostic.update_layout(
        title=f"Regression and residual diagnostic<br>{symbol} | {exchange} | Outspoken Market",
        template="plotly_white",
        width=1000,
        height=800,
        legend_title="Series"
    )

    fig_diagnostic.update_xaxes(title_text="Date", row=2, col=1)
    fig_diagnostic.update_yaxes(title_text="Price", row=1, col=1)
    fig_diagnostic.update_yaxes(title_text="Residual", row=2, col=1)

    fig_diagnostic.show()

    # ========================================================
    # 15. Final performance summary
    # ========================================================

    def summarize_strategy(data, return_column, signal_column):
        """
        Calculate simple performance statistics for a trading strategy.

        Parameters
        ----------
        data : pandas.DataFrame
            Dataset containing signals and returns.

        return_column : str
            Name of the column containing the strategy returns.

        signal_column : str
            Name of the column containing the strategy signals.

        Returns
        -------
        pandas.Series
            Strategy performance summary.
        """

        returns = data[return_column]
        signals = data[signal_column]

        # Number of periods where the strategy had an active position.
        total_trades = (signals != 0).sum()

        # Final cumulative return in percentage points.
        total_return = returns.sum()

        # Average return per period.
        average_return_per_period = returns.mean()

        # Return standard deviation per period.
        return_std_per_period = returns.std()

        # Hit rate only when there was a trade.
        #
        # This avoids counting flat periods as wins or losses.
        if total_trades > 0:
            trade_hit_rate = (returns[signals != 0] > 0).mean() * 100
        else:
            trade_hit_rate = np.nan

        return pd.Series({
            "Total trades": total_trades,
            "Total cumulative return %": total_return,
            "Average return per period %": average_return_per_period,
            "Return standard deviation per period %": return_std_per_period,
            "Trade hit rate %": trade_hit_rate
        })

    # Summary for simple rule.
    simple_summary = summarize_strategy(
        df_test,
        "Simple_Strategy_Return",
        "Simple_Rule"
    )

    # Summary for quartile rule.
    quartile_summary = summarize_strategy(
        df_test,
        "Quartile_Strategy_Return",
        "Quartile_Rule"
    )

    # Combine both summaries in a single table.
    final_summary = pd.DataFrame({
        "Simple rule": simple_summary,
        "Quartile rule": quartile_summary
    })

    print("\nFinal strategy summary")
    print("======================")
    print(final_summary.round(4))

    # ========================================================
    # 16. Return all useful objects
    # ========================================================

    # Returning objects is important because it allows the user to inspect,
    # reuse, export or plot the results later without rerunning everything.
    return {
        "data": df,
        "train": df_train,
        "test": df_test,
        "model": model,
        "residual_summary": residual_summary,
        "simple_table": simple_table,
        "quartile_table": quartile_table,
        "final_summary": final_summary,
        "parameters": {
            "symbol": symbol,
            "exchange": exchange,
            "interval": interval,
            "n_bars": n_bars,
            "train_size": train_size,
            "threshold": threshold,
            "trading_cost": trading_cost
        }
    }

In [10]:
# How to use it

results = run_regression_residual_strategy(
    symbol="VIX",
    exchange="CAPITALCOM",
    interval=Interval.in_daily,
    n_bars=5000,
    train_size=2000,
    threshold=1,
    trading_cost=0.01,
    show_model_summary=True
)


Dataset information
Asset: VIX
Exchange: CAPITALCOM
Total records after cleaning: 3636
Training records: 2000 | 2012-01-13 to 2019-12-19
Testing records: 1636 | 2019-12-20 to 2026-04-28

OLS regression summary
                            OLS Regression Results                            
Dep. Variable:                  Close   R-squared:                       0.900
Model:                            OLS   Adj. R-squared:                  0.900
Method:                 Least Squares   F-statistic:                 1.806e+04
Date:                Wed, 29 Apr 2026   Prob (F-statistic):               0.00
Time:                        15:44:35   Log-Likelihood:                -3365.0
No. Observations:                2000   AIC:                             6734.
Df Residuals:                    1998   BIC:                             6745.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         


Percentage table: Target1_Bin vs Simple_Rule
Simple_Rule     -1      0     1
Target1_Bin                    
0            18.65  72.70  8.65
1            13.67  79.89  6.43



Percentage table: Target1_Bin vs Quartile_Rule
Quartile_Rule     -1      0      1
Target1_Bin                       
0              22.13  48.54  29.33
1              16.35  55.50  28.15



Final strategy summary
                                        Simple rule  Quartile rule
Total trades                               393.0000       790.0000
Total cumulative return %                  175.3927       352.9971
Average return per period %                  0.1072         0.2158
Return standard deviation per period %       3.6337         4.1773
Trade hit rate %                            54.4529        51.5190


In [ ]:
# Then you can access the outputs like this:

df_full = results["data"]
df_train = results["train"]
df_test = results["test"]
model = results["model"]
summary = results["final_summary"]

In [11]:
#For another asset, you only change the function call:

results = run_regression_residual_strategy(
    symbol="BTCUSD",
    exchange="CAPITALCOM",
    interval=Interval.in_4_hour,
    n_bars=5000,
    train_size=2000,
    threshold=500,
    trading_cost=0.01
)


Dataset information
Asset: BTCUSD
Exchange: CAPITALCOM
Total records after cleaning: 4998
Training records: 2000 | 2024-01-16 to 2024-12-15
Testing records: 2998 | 2024-12-15 to 2026-04-29

OLS regression summary
                            OLS Regression Results                            
Dep. Variable:                  Close   R-squared:                       0.997
Model:                            OLS   Adj. R-squared:                  0.997
Method:                 Least Squares   F-statistic:                 5.877e+05
Date:                Wed, 29 Apr 2026   Prob (F-statistic):               0.00
Time:                        15:45:31   Log-Likelihood:                -16060.
No. Observations:                2000   AIC:                         3.212e+04
Df Residuals:                    1998   BIC:                         3.213e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                      


Percentage table: Target1_Bin vs Simple_Rule
Simple_Rule     -1      0      1
Target1_Bin                     
0            22.13  58.69  19.18
1            19.18  57.60  23.22



Percentage table: Target1_Bin vs Quartile_Rule
Quartile_Rule     -1      0      1
Target1_Bin                       
0              18.65  43.80  37.56
1              16.59  39.75  43.66



Final strategy summary
                                        Simple rule  Quartile rule
Total trades                              1255.0000      1746.0000
Total cumulative return %                   43.6307        19.5117
Average return per period %                  0.0146         0.0065
Return standard deviation per period %       0.7102         0.8241
Trade hit rate %                            53.7052        52.9782
